# R2: Ablation Studies — Shuffled-Rationale & Plain-Continuation

**Purpose:** Test whether rationale SEMANTICS matter or if Stage 2 improvement comes from extra training.

Two ablations (both use seed=42):
1. **Shuffled-rationale:** Stage 2 with randomly reassigned rationales (same format, wrong semantics)
2. **Plain-continuation:** Stage 2 on same 1,221 texts in classification-only format (no rationale/implied)

**Output:** `results_ablations.json`

**Estimated time:** 5-7h on T4x2 Kaggle

## Step 1: Install Dependencies

In [1]:
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.0
!pip install -q transformers datasets accelerate sentencepiece
!pip install -q scikit-learn pandas numpy tqdm openpyxl
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes
!pip install -q peft

import os, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GRPC_VERBOSITY'] = 'ERROR'
warnings.filterwarnings('ignore')
print('Dependencies installed')

Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.25 requires protobuf>=5.29.5, but you have protobuf 3.20.0 which is incompatible.
google-cloud-videointelligence 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.0 which is incompatible.
google-cloud-vision 3.12.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.0 which is incompatible.
ray

## Step 2: Setup & GPU Detection (Kaggle)

In [2]:
import sys, os, json
from pathlib import Path

# Copy source files to working directory
!cp -r /kaggle/input/vithsd-experiment/* /kaggle/working/

os.chdir('/kaggle/working')
sys.path.insert(0, '/kaggle/working')

OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

import config
config.OUTPUT_DIR = OUTPUT_DIR

print('Setup complete')

Setup complete


In [3]:
# 🎮 GPU Auto-Detection & Optimization
import torch
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"\n{'='*70}")
    print(f"🎮 DETECTED: {num_gpus} GPU(s)")
    print(f"{'='*70}")

    for i in range(num_gpus):
        gpu_name = torch.cuda.get_device_name(i)
        print(f"GPU {i}: {gpu_name}")

    if num_gpus == 1:
        # Single GPU (P100 16GB)
        print(f"\n{'='*70}")
        print("⚙️  SINGLE GPU MODE (P100 Optimized)")
        print(f"{'='*70}")

        os.environ['CUDA_VISIBLE_DEVICES'] = '0'

        BATCH_SIZE_VISOBERT = 12   # ViSoBERT ~110M params, same footprint as PhoBERT

        GPU_CONFIG = {
            'num_gpus': 1,
            'device': 'cuda:0',
            'batch_sizes': {'visobert': BATCH_SIZE_VISOBERT},
            'gradient_accumulation_steps': 2
        }

    elif num_gpus >= 2:
        # Multi-GPU (T4 x2 32GB)
        print(f"\n{'='*70}")
        print("⚡ MULTI-GPU MODE (T4 x2 Maximum Performance)")
        print(f"{'='*70}")

        BATCH_SIZE_VISOBERT = 16

        GPU_CONFIG = {
            'num_gpus': num_gpus,
            'device': 'cuda',
            'batch_sizes': {'visobert': BATCH_SIZE_VISOBERT},
            'gradient_accumulation_steps': 1
        }

    print(f"\n📊 Configured Batch Size:")
    print(f"   ViSoBERT: {GPU_CONFIG['batch_sizes']['visobert']}")
    print(f"\n{'='*70}\n")

else:
    print("❌ No GPU detected! This notebook requires GPU.")
    GPU_CONFIG = None

PyTorch version: 2.10.0+cu128
CUDA available: True

🎮 DETECTED: 2 GPU(s)
GPU 0: Tesla T4
GPU 1: Tesla T4

⚡ MULTI-GPU MODE (T4 x2 Maximum Performance)

📊 Configured Batch Size:
   ViSoBERT: 16




In [4]:
import torch, random, gc
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected!')

gpu_props = torch.cuda.get_device_properties(0)
gpu_name = gpu_props.name
gpu_mem = gpu_props.total_memory / 1e9

print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

BATCH_SIZE = 8 if gpu_mem >= 35 else (4 if gpu_mem >= 14 else 2)
SEED = 42
print(f'Batch size: {BATCH_SIZE}')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4 (15.6 GB)
Batch size: 4


## Step 3: Load Data

In [5]:
from config import FINAL_LABELS
from data_preparation import load_dataset_A
from evaluation import Evaluator
from models import create_model

train_texts, train_labels = load_dataset_A('train')
test_texts, test_labels = load_dataset_A('test')

# Load rationale data from separate Kaggle dataset
rationale_path = Path('/kaggle/input/datasets/antrnghongthnh/dataset-ie/dataset_rationale.json')
assert rationale_path.exists(), f'Missing: {rationale_path}. Add dataset-ie as Input.'

with open(rationale_path, 'r', encoding='utf-8') as f:
    rationale_data = json.load(f)

r_train_texts, r_train_implied, r_train_rationale, r_train_labels = [], [], [], []
for r in rationale_data:
    if str(r.get('dataset', '')).lower().strip() != 'train':
        continue
    content = (r.get('content') or '').strip()
    implied = (r.get('implied_statement') or '').strip()
    if not content or not implied:
        continue
    r_train_texts.append(content)
    r_train_implied.append(implied)
    r_train_rationale.append(r.get('rationale', []))
    r_train_labels.append(r.get('labels', []))

alignment_pairs = [{'content': c, 'implied': i} for c, i in zip(r_train_texts, r_train_implied)]
print(f'Train: {len(train_texts)}, Test: {len(test_texts)}, Rationale pairs: {len(alignment_pairs)}')

[ViHSDLoader] Loaded 7000 samples from train
[ViHSDLoader] Loaded 1800 samples from test
Train: 7000, Test: 1800, Rationale pairs: 1221


## Step 4: Helper Functions

In [6]:
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import time

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def stage2_train(model, training_texts, num_epochs=1, learning_rate=5e-5):
    """Generic Stage 2 training on pre-formatted texts."""
    class TextDataset(Dataset):
        def __init__(self, texts, tokenizer, max_length):
            self.encodings = tokenizer(
                texts, truncation=True, padding=True,
                max_length=max_length, return_tensors='pt'
            )
        def __len__(self):
            return len(self.encodings['input_ids'])
        def __getitem__(self, idx):
            return {
                'input_ids': self.encodings['input_ids'][idx],
                'attention_mask': self.encodings['attention_mask'][idx]
            }

    dataset = TextDataset(training_texts, model.tokenizer, model.max_length)
    dataloader = DataLoader(dataset, batch_size=model.batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.model.parameters(), lr=learning_rate)

    model.model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        progress = tqdm(dataloader, desc=f'  Stage2 Ep {epoch+1}/{num_epochs}')
        for batch in progress:
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            optimizer.zero_grad()
            outputs = model.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=input_ids
            )
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.model.parameters(), 1.0)
            optimizer.step()
            progress.set_postfix({'loss': f'{loss.item():.4f}'})
        print(f'  Epoch {epoch+1}: avg_loss = {total_loss / len(dataloader):.4f}')


def cleanup_model(model):
    if hasattr(model, 'model') and model.model is not None:
        del model.model
    if hasattr(model, 'tokenizer') and model.tokenizer is not None:
        del model.tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()


def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

print('Helper functions defined')

Helper functions defined


## Step 5: Prepare Ablation Data

**Ablation A - Shuffled-Rationale:** Each sample gets another sample's rationale/implied.
Same CoT format, but semantically unrelated rationale.

**Ablation B - Plain-Continuation:** Same 1,221 texts in standard classification format (no CoT).

In [7]:
set_seed(SEED)
n = len(r_train_texts)

# Create a derangement (no element maps to itself)
shuffled_indices = list(range(n))
random.shuffle(shuffled_indices)
for i in range(n):
    if shuffled_indices[i] == i:
        j = (i + 1) % n
        shuffled_indices[i], shuffled_indices[j] = shuffled_indices[j], shuffled_indices[i]

shuffled_implied = [r_train_implied[shuffled_indices[i]] for i in range(n)]
shuffled_rationale = [r_train_rationale[shuffled_indices[i]] for i in range(n)]

same_count = sum(1 for i in range(n) if shuffled_indices[i] == i)
print(f'Shuffled: {n} pairs, {same_count} self-matches (should be 0)')
print(f'Plain: {n} texts in classification format')

Shuffled: 1221 pairs, 0 self-matches (should be 0)
Plain: 1221 texts in classification format


## Step 6: Ablation A — Shuffled-Rationale

In [8]:
print(f'{"="*70}')
print('ABLATION A: Shuffled-Rationale Stage 2')
print(f'{"="*70}')
t0 = time.time()

set_seed(SEED)
evaluator = Evaluator()

model_shuf = create_model(
    'qwen', dataset_type='A',
    batch_size=BATCH_SIZE, num_epochs=2,
    learning_rate=2e-4, lora_r=8, lora_alpha=16
)

# Stage 1 (identical to RSQwen)
print('  Stage 1...')
set_seed(SEED)
model_shuf.train(train_texts, train_labels)

# Stage 2 with SHUFFLED rationale
print('  Stage 2 (shuffled rationale)...')
set_seed(SEED)

shuffled_texts = []
for i in range(len(r_train_texts)):
    input_text = model_shuf._format_input_cot(r_train_texts[i])
    output_text = model_shuf._format_output_cot(
        r_train_labels[i],       # ORIGINAL labels
        shuffled_rationale[i],    # SHUFFLED rationale
        shuffled_implied[i]       # SHUFFLED implied
    )
    shuffled_texts.append(input_text + output_text)

stage2_train(model_shuf, shuffled_texts, num_epochs=1, learning_rate=5e-5)

print('  Predicting...')
pred_shuf, raw_shuf = model_shuf.predict(test_texts)
res_shuf = evaluator.evaluate(test_labels, pred_shuf, 'Shuffled-Rationale', 'ablation')

elapsed = (time.time() - t0) / 60
print(f'  DONE in {elapsed:.1f} min | F1-Micro={res_shuf["f1_micro"]:.4f}, F1-Macro={res_shuf["f1_macro"]:.4f}')

cleanup_model(model_shuf)

ABLATION A: Shuffled-Rationale Stage 2
  Stage 1...
[Qwen2.5-3B_MultiLabel] Training with QLoRA (4-bit)...
  Model: Qwen/Qwen2.5-3B-Instruct
  Device: cuda
  Original samples: 7000
  Mode: Standard (Dataset A)
  [Oversampling] Added 540 samples for minority labels
  LoRA config: r=8, alpha=16


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Epoch 2: avg_loss = 0.1906
  ✓ Training completed
  Stage 2 (shuffled rationale)...



  Stage2 Ep 1/1: 100%|██████████| 306/306 [28:20<00:00,  5.56s/it, loss=0.3976]


  Epoch 1: avg_loss = 0.5746
  Predicting...



Predicting:   0%|          | 0/1800 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Predicting: 100%|██████████| 1800/1800 [34:21<00:00,  1.15s/it]


  DONE in 341.4 min | F1-Micro=0.5838, F1-Macro=0.2952


## Step 7: Ablation B — Plain-Continuation

In [9]:
print(f'{"="*70}')
print('ABLATION B: Plain-Continuation Stage 2')
print(f'{"="*70}')
t0 = time.time()

set_seed(SEED)
evaluator = Evaluator()

model_plain = create_model(
    'qwen', dataset_type='A',
    batch_size=BATCH_SIZE, num_epochs=2,
    learning_rate=2e-4, lora_r=8, lora_alpha=16
)

# Stage 1 (identical to RSQwen)
print('  Stage 1...')
set_seed(SEED)
model_plain.train(train_texts, train_labels)

# Stage 2 with classification-only format (NO rationale/implied)
print('  Stage 2 (plain continuation, no CoT)...')
set_seed(SEED)

plain_texts = []
for i in range(len(r_train_texts)):
    input_text = model_plain._format_input_standard(r_train_texts[i])
    output_text = model_plain._format_output_standard(r_train_labels[i])
    plain_texts.append(input_text + output_text)

stage2_train(model_plain, plain_texts, num_epochs=1, learning_rate=5e-5)

print('  Predicting...')
pred_plain, raw_plain = model_plain.predict(test_texts)
res_plain = evaluator.evaluate(test_labels, pred_plain, 'Plain-Continuation', 'ablation')

elapsed = (time.time() - t0) / 60
print(f'  DONE in {elapsed:.1f} min | F1-Micro={res_plain["f1_micro"]:.4f}, F1-Macro={res_plain["f1_macro"]:.4f}')

cleanup_model(model_plain)

ABLATION B: Plain-Continuation Stage 2
  Stage 1...
[Qwen2.5-3B_MultiLabel] Training with QLoRA (4-bit)...
  Model: Qwen/Qwen2.5-3B-Instruct
  Device: cuda
  Original samples: 7000
  Mode: Standard (Dataset A)
  [Oversampling] Added 540 samples for minority labels
  LoRA config: r=8, alpha=16


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827
  Training for 2 epochs...



Epoch 1/2: 100%|██████████| 1885/1885 [2:19:25<00:00,  4.44s/it, loss=0.2015]


  Epoch 1: avg_loss = 0.2495


...

  Epoch 2: avg_loss = 0.1904
  ✓ Training completed
  Stage 2 (plain continuation, no CoT)...



  Stage2 Ep 1/1: 100%|██████████| 306/306 [22:32<00:00,  4.42s/it, loss=0.0519]


  Epoch 1: avg_loss = 0.1918
  Predicting...



Predicting: 100%|██████████| 1800/1800 [48:48<00:00,  1.63s/it]


  DONE in 350.1 min | F1-Micro=0.3635, F1-Macro=0.2890


## Step 8: Summary & Save

In [10]:
print(f'{"="*70}')
print(f'{"ABLATION RESULTS SUMMARY":^70}')
print(f'{"="*70}')
print(f'{"":>22} {"F1-Micro":>12} {"F1-Macro":>12} {"Accuracy":>12}')
print(f'{"-"*60}')
print(f'{"Shuffled-Rationale":>22} {res_shuf["f1_micro"]*100:10.2f}% {res_shuf["f1_macro"]*100:10.2f}% {res_shuf["subset_accuracy"]*100:10.2f}%')
print(f'{"Plain-Continuation":>22} {res_plain["f1_micro"]*100:10.2f}% {res_plain["f1_macro"]*100:10.2f}% {res_plain["subset_accuracy"]*100:10.2f}%')
print(f'\n(Compare with RSQwen and Vanilla from R1 results_multiseed.json)')

ablation_results = {
    'seed': SEED,
    'shuffled_rationale': convert_to_serializable(res_shuf),
    'plain_continuation': convert_to_serializable(res_plain),
    'shuffled_predictions': pred_shuf,
    'plain_predictions': pred_plain
}

with open(OUTPUT_DIR / 'results_ablations.json', 'w') as f:
    json.dump(ablation_results, f, indent=2, ensure_ascii=False)

print(f'\nSaved: {OUTPUT_DIR / "results_ablations.json"}')
print('DONE! Download results_ablations.json from Drive.')

                       ABLATION RESULTS SUMMARY                       
                           F1-Micro     F1-Macro     Accuracy
------------------------------------------------------------
    Shuffled-Rationale      58.38%      29.52%      52.11%
    Plain-Continuation      36.35%      28.90%      25.78%

(Compare with RSQwen and Vanilla from R1 results_multiseed.json)

Saved: /kaggle/working/outputs/results_ablations.json
DONE! Download results_ablations.json from Drive.
